# Finetune Pretrained YOLOv8 on Custom Dataset

Objectives: by the end of this tutorial you shall be able to:

- Finetune pretrained YOLOv8 on **Custom Dataset**
- Configure [training settings](https://docs.ultralytics.com/modes/train/#train-settings) including [image augmentations](https://docs.ultralytics.com/modes/train/#augmentation-settings-and-hyperparameters) and layer freezing (to fine-tune only specific layers).
- Auto hyperparameter search using yolo's `.tune()` method, and copy that into model train settings.
- Resume training from last checkpoint
- Export (save) model with proper optimization flags, for later inference
- Leave a long-running model training:
  - schedule training to stop after a specified number of epochs, or after a specified duration (in hours)
  - programmatically download model weights (`best.pt`) and shutdown Colab run-time to (to preserve compute units)

References:

- This tutorial makes use of: [**How to Train YOLOv8 Object Detection on a Custom Dataset**](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolov8-object-detection-on-custom-dataset.ipynb) - roboflow/notebooks

## Dataset: Roboflow Universe

Need data for your project? Before spending time on annotating, check out Roboflow Universe, a repository of more than 110,000 open-source datasets that you can use in your projects. You'll find datasets containing everything from annotated cracks in concrete to plant images with disease annotations.

[![Roboflow Universe](https://media.roboflow.com/notebooks/template/uni-banner-frame.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672878480290)](https://universe.roboflow.com/)

## Preparing a custom dataset

Building a custom dataset can be a painful process. It might take dozens or even hundreds of hours to collect images, label them, and export them in the proper format. Fortunately, Roboflow makes this process as straightforward and fast as possible. Let me show you how!

### Step 1: Creating project

Before you start, you need to create a Roboflow [account](https://app.roboflow.com/login). Once you do that, you can create a new project in the Roboflow [dashboard](https://app.roboflow.com/). Keep in mind to choose the right project type. In our case, Object Detection.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/creating-project.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672929799852"
  >
</div>

### Step 2: Uploading images

Next, add the data to your newly created project. You can do it via API or through our [web interface](https://docs.roboflow.com/adding-data/object-detection).

If you drag and drop a directory with a dataset in a [supported format](https://roboflow.com/formats), the Roboflow dashboard will automatically read the images and annotations together.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/uploading-images.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672929808290"
  >
</div>

### Step 3: Labeling

If you only have images, you can label them in [Roboflow Annotate](https://docs.roboflow.com/annotate).

<div align="center">
  <img
    width="640"
    src="https://user-images.githubusercontent.com/26109316/210901980-04861efd-dfc0-4a01-9373-13a36b5e1df4.gif"
  >
</div>

### Step 4: Generate new dataset version

Now that we have our images and annotations added, we can Generate a Dataset Version. When Generating a Version, you may elect to add preprocessing and augmentations. This step is completely optional, however, it can allow you to significantly improve the robustness of your model.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/generate-new-version.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1673003597834"
  >
</div>

### Step 5: Exporting dataset

Once the dataset version is generated, we have a hosted dataset we can load directly into our notebook for easy training. Click `Export` and select the `YOLO v8` dataset format. (Formerly, we used to use `Yolov5`, as the gif shows)

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/export.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672943313709"
  >
</div>


## Back to YOLO

### Step 1. Select a GPU Runtime

- CPU would take forever to train deep learning models. Exported models, however, can still run on CPU.
- Current Colab allows you to select a runtime from the toolbar:
  - `Runtime > Change runtime type` and select something with GPU or TPU in it.

In [ ]:
import glob
from IPython.display import Image, display

### Step 2. Install and import YOLOv8 from Ultralytics

Ultralytics is a platform for deep learning. They offer newer versions of yolo almost every year. We will use yolov8 since it is stable.

In [ ]:

!pip install ultralytics==8.0.196 -q

import ultralytics
ultralytics.checks()

from ultralytics import YOLO

### Step 3. Configure Train Settings

- First, we want the default config, and we want to rename it to `config.yaml`.

In [ ]:
!yolo copy-cfg
!mv default_copy.yaml config.yaml

The [`config.yaml`](https://docs.ultralytics.com/modes/train/#train-settings) file includes:
- **Train settings**
- **Val/Test settings**
- **Prediction settings**
- **Export settings**
- **Hyperparameters** (including augmentations)
- **Tracker settings**

We mostly care about **Hyperparameters**. We will specify other arguments in the CLI directly.

### Fine-tune Last N Layers

**Transfer learning** is a useful way to quickly retrain a model on new data without having to retrain the entire network. This requires less resources than normal training and allows for faster training times.

The Argument `freeze` (default: `None`) Freezes the first N layers of the model or specified layers by index, reducing the number of trainable parameters.

To specify `freeze: N` we need to look at: [cfg/models/v8/yolov8.yaml](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/models/v8/yolov8.yaml) which shows yolov8 architecture (backbone, intermediate layers, and head).

We notice that:

- if we were to freeze just the backbone, then `freeze: 9`.
- if we were to also freeze the head entirely but leaving the last layer, then `freeze: 21`

You can also specify it in the `config.yaml` if you wish to do so, like so:

`config.yaml`:

```yaml
freeze: 21
```

### Step 3. Load Custom Dataset from Roboflow


#### Sidenote: Python variables can be accessed from shell

In [ ]:
import os

HOME = os.getcwd()
print(HOME)

In [ ]:
!echo $HOME {HOME}

We will download football players dataset just for demonstration.

In [ ]:
# Create directory and hop in to make the `download` method do it there
!mkdir {HOME}/datasets
%cd {HOME}/datasets

!pip install roboflow --quiet

from roboflow import Roboflow

rf = Roboflow(api_key='ROBOFLOW_API_KEY')
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(1).download("yolov8")

In [ ]:
%cd {HOME}

In [ ]:
print(dataset.location)

You will see you have `datasets/.../data.yaml` file, with similar content like:

```yaml
names:
- ball
- goalkeeper
- player
- referee
nc: 4

test: test/images
train: train/images
val: valid/images
```

Make sure the paths are correct, for keys: **test**, **train**, and **val**.


💡 Tip: Always train from a local dataset (colab session). Mounted or network drives like Google Drive will be **very slow**.

### Step 4. Train the Model

See: [CLI Usage](https://docs.ultralytics.com/usage/cli/) to learn how the commands are written.

We specify the `cfg=config.yaml` (configuration file) and specify more arugments through the CLI directly:

- `mode: train`
- `epochs: 5` -- increase this
- `freeze: 21` -- we may decrease this (or remove it to fine-tune the whole model)
- `model: yolov8n.pt` -- we may [select bigger models](https://docs.ultralytics.com/models/yolov8/#supported-tasks-and-modes)
- `data: {dataset.location}/data.yaml`

Note: in `config.yaml` it is specified that model weights are saved:

```yaml
save: True  # (bool) save train checkpoints and predict results
```

In [ ]:
!yolo task=detect mode=train cfg={HOME}/config.yaml data={dataset.location}/data.yaml \
  model=yolov8n.pt epochs=2 freeze=21 plots=True

For full details: [Checkout yolov8 supported tasks and modes](https://docs.ultralytics.com/models/yolov8/#supported-tasks-and-modes).

Be sure to choose the right **model** for the right **task**, and the right **dataset**:

- `task: detect` (`model=yolov8n.pt`) Object Detection
- `task: segment` (`model=YOLOv8n-seg.pt`) Instance Segmentation
- `task: classify` (`model=YOLOv8n-cls.pt`) Classification

Note: the extra `n` indicates the size: Nano.

And make sure you choose the right **mode**:

- `mode: train`
- `mode: predict`
- `mode: val` -- evaluate
- `mode: export` -- save model in optimized format for later inference


### Step 5. Evaluate Model

See: [modes/val](https://docs.ultralytics.com/modes/val/#usage-examples) for full details.

- Let's find our model best weights at `runs/detect/train<X>/weights/best.pt` and run `val` to evaluate on the validation set:
- Note: resulsts are saved to `runs/detect/val<Y>`

In [ ]:
!yolo task=detect mode=val \
  model=runs/detect/train2/weights/best.pt \
  data={dataset.location}/data.yaml \
  plots=True

We can also look at the `results.png` for plots, or specifically at individual plots, or the `confusion_matrix.png` plot:

In [ ]:
# import os
# print([f for f in os.listdir(f'{HOME}/runs/detect/train2') if f.endswith('.png') or f.endswith('.jpg')])

Image(filename=f'{HOME}/runs/detect/train2/results.png', width=640)
Image(filename=f'{HOME}/runs/detect/train2/confusion_matrix.png', width=640)

We can also look at a batch of predictions by the model either in `train` or `val` sets:

In [ ]:
Image(filename=f'{HOME}/runs/detect/train2/val_batch1_pred.jpg', width=640)

We can also run predictions on the `test` set, and look at the results. (Notice the `mode=predict`):


In [ ]:
!yolo task=detect mode=predict model=runs/detect/train2/weights/best.pt \
  source={dataset.location}/test/images

In [ ]:
for image_path in glob.glob(f'{HOME}/runs/detect/predict/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))
      print("\n")

To run `predict` **on video**, read the docs for [`source`](https://docs.ultralytics.com/modes/predict/#inference-sources).

### Step 6. Export Model

The `export` commands saves the model for later inference (not further training) in an optimized form. Example:

- Export to [ONNX](https://docs.ultralytics.com/integrations/onnx/) or OpenVINO formats for up to **3x CPU speedup**.
- Export to [TensorRT](https://docs.ultralytics.com/integrations/tensorrt/) format for up to **5x GPU speedup**.

In [ ]:
!yolo task=detect mode=export model=runs/detect/train/weights/best.pt \
  format=onnx simplify=True half=True

You can later load the model and use it to `predict`:

In [ ]:
!yolo predict model=yolov8n.onnx source='https://ultralytics.com/images/bus.jpg'

See [modes/export](https://docs.ultralytics.com/modes/export/) for full details on export arguments and formats.

**Looking at the `mAP50` and `mAP50-95`, let's assume we conclude that our model needs improvement.**

### Improving the Model

We can always train for **more epochs**:

In [ ]:
!yolo task=detect mode=train cfg=config.yaml \
  data={dataset.location}/data.yaml \
  model=runs/detect/train5/weights/last.pt \
  epochs=5 \
  freeze=21

[**Hyperparamter Search**](https://docs.ultralytics.com/guides/hyperparameter-tuning/) via `model.tune` and look at the output at `runs/detect/tune` to copy and paste these into `config.yaml`.

**Note: hyperparameter tuning can be computationally intensive.** So, don't exhaust your compute units.



In [ ]:
model = YOLO("./yolov8n.pt") # make sure this is our model
model.tune(data=f"{dataset.location}/data.yaml", epochs=5, iterations=10,
  optimizer="AdamW", plots=False, save=False, val=False)

Training directories like `train1/` contain individual tuning iterations, i.e. one model trained with one set of hyperparameters.

The `tune/` directory contains tuning results from all the individual model trainings:

```
runs/
└── detect/
    ├── train1/
    ├── train2/
    ├── ...
    └── tune/
        ├── best_hyperparameters.yaml
        ├── best_fitness.png
        ├── tune_results.csv
        ├── tune_scatter_plots.png
        └── weights/
            ├── last.pt
            └── best.pt
```

Other Hyperparameters Tuning:

- You may play around with `batch` size or set it to `-1` for **AutoBatch**ing.

- You may try different image sizes (`imgsz`) and observe which works best with the model.

- You may also experiment with other hyperparameters, like: `label_smoothing`, or augmentation values that are currently set to `0`.

- You can also experiment with first `freeze: 21` for few epochs, then unfreezing gradually: `freeze: 15` with few more epochs, and so on, until you `freeze: None` (fine-tuning the whole model).

- Since `model: yolov8n.pt` (`n` for nano), you may [select a bigger model](https://docs.ultralytics.com/models/yolov8/#supported-tasks-and-modes)



[**Augmentation**](https://docs.ultralytics.com/modes/train/#augmentation-settings-and-hyperparameters) techniques are essential for improving the robustness and performance of YOLO models by introducing variability into the training data, helping the model generalize better to unseen data.

In `config.yaml`, under **Hyperparameters** you would find (down a bit):

```yaml
hsv_h: 0.015  # (float) image HSV-Hue augmentation (fraction)
hsv_s: 0.7  # (float) image HSV-Saturation augmentation (fraction)
hsv_v: 0.4  # (float) image HSV-Value augmentation (fraction)
degrees: 0.0  # (float) image rotation (+/- deg)
translate: 0.1  # (float) image translation (+/- fraction)
scale: 0.5  # (float) image scale (+/- gain)
shear: 0.0  # (float) image shear (+/- deg)
perspective: 0.0  # (float) image perspective (+/- fraction), range 0-0.001
flipud: 0.0  # (float) image flip up-down (probability)
fliplr: 0.5  # (float) image flip left-right (probability)
mosaic: 1.0  # (float) image mosaic (probability)
mixup: 0.0  # (float) image mixup (probability)
copy_paste: 0.0  # (float) segment copy-paste (probability)
```

Assuming we have done the above and change some of:
1. Hyperparameters based on fine-tuning via `.tune` method results in `best_hyperparameters.yaml`
2. Other hyperparameters like `batch`, `epochs`, and `imgsz`
3. Augmentations

Let's now continue our training with our newly set train settings (`config.yaml`):

In [ ]:
!yolo task=detect mode=train cfg=config.yaml \
  model=runs/detect/train2/weights/best.pt \
  data={dataset.location}/data.yaml \
  epochs=5 imgsz=640 freeze=21

#### Extra: Real-time logs with Tensorboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {HOME}/runs

## Extra: how to start long-running training and schedule a stop time, while downloading and not losing your work

### How to Terminate Colab Sessions Programmatically

You can [terminate Colab sessions programmatically](https://twitter.com/GoogleColab/status/1569389018311426051?lang=en) with the following code:

```
from google.colab import runtime
runtime.unassign()
```

The `unassign()` function marks the currently connected runtime for deletion and disconnects any notebook sessions.


### How to Download file Programmatically

```py
from google.colab import files
files.download('example.txt')
```

In [ ]:
from google.colab import files
files.download(f'{HOME}/runs/detect/train2/weights/best.pt')

### How to Save the model on your mounted drive

```python
# mount drive
from google.colab import drive
drive.mount('/content/drive')
```

```sh
# copy best.pt inside drive
!cp {HOME}/runs/detect/train2/weights/best.pt /content/drive/MyDrive/best.pt
```

**Combine `files.download()` (or the `cp` commmand) and `runtimne.unassign()` after training for a number of `epochs` or specific `time` in hours (both can be set in `data.yaml`) to save compute units.**

- `epochs` (default: 100) - Total number of training epochs. Each epoch represents a full pass over the entire dataset. Adjusting this value can affect training duration and model performance.

- `time` (default: None) - Maximum training time in hours. If set, this overrides the epochs argument, allowing training to automatically stop after the specified duration. Useful for time-constrained training scenarios.